# Parallel Multi-GPU Training on Kaggle T4×2

Two **independent single-GPU processes** running side-by-side, one per GPU.
No `DataParallel`, no DDP — each session pins itself to one GPU via
`CUDA_VISIBLE_DEVICES` and writes to its own output directory.

Why this and not DDP?

- DDP shines when one model + one batch is split across GPUs. We're doing the
  opposite — *different* models / configs in parallel.
- DDP launch failure modes are hostile (NCCL hangs, sync barriers); two
  independent processes have none of that surface.
- Each run can fail / be cancelled / restart independently.

## TL;DR

**Pattern A** — the recommended path. Open this notebook in **two separate
Kaggle sessions** from the same repo. Each session pins itself to one GPU.
Logs stay tidy, monitoring is trivial, and a crash in one session doesn't
kill the other. Use Pattern A by default.

**Pattern B** — one notebook spawns two subprocesses. Documented at the
bottom for tightly-coupled experiments. Logs interleave, harder to monitor,
single point of failure — you've been warned.

## 1. Smoke test — verify the environment

Run this first to confirm CUDA is wired up and `CUDA_VISIBLE_DEVICES` is
doing what you expect.

In [ ]:
import os, torch

print(f'PyTorch:            {torch.__version__}')
print(f'CUDA available:     {torch.cuda.is_available()}')
print(f'GPU count:          {torch.cuda.device_count()}')
print(f'CUDA_VISIBLE_DEVICES: {os.environ.get("CUDA_VISIBLE_DEVICES", "unset")}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  [{i}] {torch.cuda.get_device_name(i)}')
if torch.cuda.device_count() < 2:
    print('\nWARNING: less than 2 GPUs visible — parallel pattern won\'t help here.')
    print('         (This is expected in a single-session view of Pattern A;\n'
          '         each session sees only its pinned GPU.)')

## 2. Pattern A — two Kaggle sessions, one GPU each (RECOMMENDED)

### Setup (do once per pair of sessions)

1. Pick the **T4×2** accelerator (Settings → Accelerator).
2. Save this notebook. Then **duplicate it** in your Kaggle account
   (“Copy & Edit”) so you have two independent notebook instances of the
   same code. Open both side-by-side in two browser tabs.
3. In each session, decide which GPU it owns:
   - Session 1 → GPU 0 (set `CUDA_VISIBLE_DEVICES="0"`)
   - Session 2 → GPU 1 (set `CUDA_VISIBLE_DEVICES="1"`)

### Critical: pin the GPU BEFORE importing torch

`CUDA_VISIBLE_DEVICES` only takes effect when set **before** the CUDA
runtime initialises. The first `import torch` (or `torch.cuda.*` call)
freezes the device list. So **the very first executed cell in each
session** must be the GPU-pinning cell, *before* anything else imports
torch.

### Session 1 — GPU 0 (uncomment in session 1)

In [ ]:
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # session 1 -> GPU 0
# MUST be the first cell executed; before any torch import.
pass

### Session 2 — GPU 1 (uncomment in session 2)

In [ ]:
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'  # session 2 -> GPU 1
# MUST be the first cell executed; before any torch import.
pass

### Verify the pin took effect

Each session should see exactly one GPU.

In [ ]:
import os, torch
n = torch.cuda.device_count()
print('Visible GPUs:', n)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', 'unset'))
if n != 1:
    raise RuntimeError(
        f'Expected exactly 1 visible GPU after pin, got {n}. '
        'Did you run the pin cell BEFORE any torch import?'
    )
print('OK — device 0 (this session\'s only GPU):', torch.cuda.get_device_name(0))

## 3. Clone the repo + install deps (each session)

Each session works in its own `/kaggle/working/` so the clones don't
collide. Both sessions install identical deps from `requirements.txt`.

In [ ]:
REPO_URL = 'https://github.com/whothemann/massmind-segmentation.git'  # EDIT if forked
REPO_BRANCH = 'main'
REPO_DIR = '/kaggle/working/massmind-segmentation'

import os, subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

In [ ]:
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt
import torch
assert torch.cuda.is_available(), 'CUDA broke after pip install'
print('CUDA still works:', torch.cuda.get_device_name(0))

In [ ]:
# Idempotent: data + splits + stats. Skip-fast if already on disk.
!python scripts/download.py
!python -m src.splits
!python -m src.stats

## 4. Launch the training run for THIS session

Edit `MODEL_ARGS` for the config this session should train. Suggested
split for the three-way ablation across two sessions:

| session | model config                       | flags |
|---------|------------------------------------|-------|
| 1 (GPU 0) | pretrained VGG16 baseline        | `--model unet_vgg16 --loss focal` |
| 2 (GPU 1) | custom_lwir + aux heads          | `--model custom_lwir --aux-heads --loss focal` |

Run the **scratch-VGG16** config (`--model unet_vgg16 --no-pretrained
--loss focal`) in whichever session finishes first.

Notes:

- `NUM_WORKERS=2` (not 4) because both sessions share the Kaggle CPU
  budget. Using 4 in each session contends and slows both down.
- `--warmup-frac` is auto-defaulted: 0.05 for `custom_lwir` and for
  `--no-pretrained`, 0.0 for pretrained VGG16. You can override with
  `--warmup-frac 0.10` if needed.
- The trainer auto-generates an output dir under `runs/` with a
  `pid<PID>` suffix — so the two sessions can't collide even if they
  finish in the same second.

In [ ]:
# EDIT THIS BLOCK PER SESSION.
EPOCHS       = 50
BATCH_SIZE   = 8
NUM_WORKERS  = 2   # shared CPUs across both sessions
AUGMENTATION = 'A'
LOSS         = 'focal'
SEED         = 42

# Pick exactly ONE of these per session. Comment out the others.
MODEL_ARGS = [
    # Session 1 (GPU 0): pretrained VGG16 baseline
    '--model', 'unet_vgg16',
]
# MODEL_ARGS = [
#     # Session 2 (GPU 1): custom LWIR architecture
#     '--model', 'custom_lwir',
#     '--aux-heads',
# ]
# MODEL_ARGS = [
#     # Scratch VGG16 (whichever session finishes first):
#     '--model', 'unet_vgg16',
#     '--no-pretrained',
# ]
print('Config:', MODEL_ARGS)

In [ ]:
import subprocess
cmd = [
    'python', '-u', '-m', 'src.train',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--augmentation', AUGMENTATION,
    '--loss', LOSS,
    '--seed', str(SEED),
    '--amp',
    '--device', 'cuda',
    *MODEL_ARGS,
]
print('Launching:')
print(' ', ' '.join(cmd))
print()
subprocess.run(cmd, check=True)

## 5. Pattern B — one notebook, two subprocesses (advanced; not recommended for iteration)

Use this only when:

- Both runs must start *exactly* simultaneously (e.g. comparing wall-clock).
- You have a tight wrapper around the run pair (analysis, ensemble eval).

Downsides:

- Logs from both processes interleave unless redirected; tail-following is
  fiddly.
- If the controlling notebook hangs / OOMs / loses its kernel, both runs die.
- Harder to interactively monitor or cancel one run.
- A bug in cell ordering can leave a child process behind on the GPU —
  you'll need `!nvidia-smi` + `kill` to recover.

For day-to-day iteration, **use Pattern A instead**.

In [ ]:
import os, subprocess
from pathlib import Path

Path('runs').mkdir(exist_ok=True)

def launch(gpu_id: int, args: list[str]) -> subprocess.Popen:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    log_path = f'runs/parallel_gpu{gpu_id}.log'
    print(f'  GPU {gpu_id}: logging to {log_path}')
    return subprocess.Popen(
        ['python', '-u', '-m', 'src.train', *args],
        env=env,
        stdout=open(log_path, 'w'),
        stderr=subprocess.STDOUT,
    )

shared_args = [
    '--epochs', '50', '--batch-size', '8', '--num-workers', '2',
    '--loss', 'focal', '--seed', '42', '--amp', '--device', 'cuda',
]

print('Launching pair of runs:')
p0 = launch(0, shared_args + ['--model', 'unet_vgg16'])
p1 = launch(1, shared_args + ['--model', 'custom_lwir', '--aux-heads'])

print('\nWaiting for both to complete (tail the logs in another cell to monitor)…')
rc0 = p0.wait()
rc1 = p1.wait()
print(f'GPU 0 exit code: {rc0}')
print(f'GPU 1 exit code: {rc1}')
if rc0 != 0 or rc1 != 0:
    print('WARNING: at least one run failed. Check the log files in runs/.')

In [ ]:
# Optional: tail the logs from another cell while the wait() above is
# running. Run in a separate browser tab so the wait cell stays unblocked.
!tail -n 30 runs/parallel_gpu0.log
print('---')
!tail -n 30 runs/parallel_gpu1.log